**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Random Matrix Theory

What do the eigenvalues of a *random* matrix look like? Not random at all — they obey laws as sharp as the CLT, and those laws decide when [covariance estimation](../../Intro_DSP/Statistical_Signal_Processing.ipynb), [MUSIC](../../Intro_DSP/Array_Processing.ipynb), and PCA can be trusted. Three sessions: the semicircle, Marchenko–Pastur (with the analytic edges verified), and the spiked-model detection threshold — the phase transition every array processor should know by heart.

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3–S4; [Concentration](../Concentration/Concentration_Inequalities.ipynb) for the 'why so deterministic' intuition.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *The Semicircle Law* (~35 min)
**Goal:** eigenvalues of symmetric random matrices: individually random, collectively deterministic.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S3. &nbsp; **Feeds into:** Session 2 (Marchenko–Pastur).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Semicircle Law</b></summary>

**Timing (~35 min).** 10 min the paradox · 10 min why it self-averages · 10 min the demo · 5 min eigenvalue repulsion.

**Open with the paradox, and let it sit.** Fill a matrix with pure noise — nothing is planned, every entry is random. Ask what its eigenvalues look like. The instinct is "random, obviously." They are not: the *histogram* converges to an exact deterministic shape with no randomness left at all. Individually random, collectively deterministic. That tension is the hook, and it is worth stating as a genuine surprise before resolving it.

**Then resolve it with machinery the room already has.** This is the [law of large numbers](../Analysis/Independence.ipynb) in matrix form. Each eigenvalue depends on *all* $n^2/2$ entries, and no single entry can move it much — which is precisely the bounded-differences condition behind [McDiarmid](../Concentration/Concentration_Inequalities.ipynb). So the spectrum self-averages, and it does so within a single draw, because one $2000\times2000$ matrix already contains two million independent numbers. Framing "one matrix is already an ensemble" is what makes the demo's result feel inevitable rather than magical.

**Point at the $1/\sqrt{n}$ scaling and ask why it is there.** Without it the eigenvalues would grow like $\sqrt{n}$ and there would be no limit shape to converge to. The scaling is what makes the question well-posed — the same role $1/\sqrt{n}$ plays in the CLT, and worth naming as the same normalisation rather than a coincidence.

**Eigenvalue repulsion is worth its five minutes.** Eigenvalues of a random symmetric matrix do not merely fail to cluster — they actively *repel*, so near-collisions are far rarer than they would be for independent points. That is why the histogram is smooth rather than clumpy, and it is a genuinely non-classical statistical behaviour. If anyone asks how far this goes: the same local statistics show up in the zeros of the Riemann zeta function and in nuclear energy levels, which is one of the strangest empirical coincidences in mathematics.

**Ask the room.** "The semicircle is supported on $[-2, 2]$. What fraction of eigenvalues should lie outside?" Zero in the limit — and the printed check reports exactly 0.0000, with the largest eigenvalue at 1.993 against a limit of 2.0. The edge is *hard*, not a soft tail, which is what makes Sessions 2 and 3's detection thresholds sharp rather than gradual.
</details>

## 2. Order from Chaos

💡 **Intuition.** Fill a symmetric matrix with i.i.d. noise, scale by $1/\sqrt{n}$, and its eigenvalue *histogram* converges to a fixed shape — Wigner's semicircle — with no randomness left in the limit. Same magic as the [LLN](../Analysis/Independence.ipynb): each eigenvalue depends on *all* $n^2/2$ entries, no single entry matters ([McDiarmid!](../Concentration/Concentration_Inequalities.ipynb)), so the ensemble self-averages. Eigenvalues also *repel* each other — near-collisions are rare — which is why the histogram is smooth, not clumpy.

In [2]:
n_dim = 2000
M = rng.standard_normal((n_dim, n_dim))
W = (M + M.T) / np.sqrt(2 * n_dim)
eigs = np.linalg.eigvalsh(W)

x = np.linspace(-2.2, 2.2, 300)
semicircle = np.where(np.abs(x) <= 2, np.sqrt(np.maximum(4 - x**2, 0)) / (2*np.pi), 0)
plt.figure(figsize=(7.5, 2.8))
plt.hist(eigs, bins=80, density=True, alpha=0.6, label=f"one {n_dim}×{n_dim} draw")
plt.plot(x, semicircle, "k", linewidth=2, label="Wigner semicircle (exact limit)")
plt.legend(); plt.title("ONE random matrix, and the histogram is already the law")
plt.tight_layout(); plt.show()
print(f"edge check: max eigenvalue {eigs.max():.3f} (limit: 2.0);  fraction outside [−2,2]: {(np.abs(eigs)>2).mean():.4f}")

edge check: max eigenvalue 1.993 (limit: 2.0);  fraction outside [−2,2]: 0.0000


/tmp/ipykernel_2981573/4081657132.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** **One** random matrix — not an average over many — and its eigenvalue histogram already lies on Wigner's semicircle. Largest eigenvalue **1.993** against a theoretical edge of exactly 2.0, and the fraction of eigenvalues outside $[-2,2]$ is **0.0000**.

**The paradox and its resolution.** Every entry was drawn independently at random; nothing was designed. Yet the *collective* behaviour of the eigenvalues has no randomness left in it. The reason is that a single $2000 \times 2000$ symmetric matrix already contains about two million independent numbers, and **each eigenvalue depends on all of them** while being insensitive to any one. That is exactly the bounded-differences condition behind [McDiarmid's inequality](../Concentration/Concentration_Inequalities.ipynb), so the spectrum concentrates — the [law of large numbers](../Analysis/Independence.ipynb) acting on a matrix rather than on a sequence.

The practical consequence is worth naming: **one draw is already an ensemble.** You do not need to average over many random matrices to see the law, which is why this demo works at all and why random matrix predictions are usable on single real datasets.

**Note the $1/\sqrt{n}$ scaling.** Without it the eigenvalues would grow like $\sqrt{n}$ and no limiting shape would exist. It is the same normalisation that makes the central limit theorem well-posed, playing the same role for the same reason.

**And note that the edge is hard.** Not one eigenvalue in 2000 strayed outside $[-2,2]$, and the maximum came within 0.007 of the boundary. The semicircle does not have tails that fade out — it *stops*. That sharpness is what makes Sessions 2 and 3 useful: if the noise had soft tails, "is this eigenvalue too big to be noise?" would be a matter of degree. Because the edge is hard, it becomes a genuine threshold.

**One phenomenon the histogram hides.** Eigenvalues of a random symmetric matrix *repel* one another — near-collisions are far rarer than for independent random points, which is why the histogram is smooth rather than clumpy. This local behaviour turns out to match the spacing statistics of nuclear energy levels and, empirically, the zeros of the Riemann zeta function. Not needed for what follows, but it is a fair indication that these laws are deeper than a convenience for signal processing.

---
### 🕐 Session 2 of 3 — *Marchenko–Pastur: the Law of Sample Covariance* (~40 min)
**Goal:** what eigenvalues of pure-noise covariance look like — and why high-dimensional PCA lies.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (spiked models).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Marchenko–Pastur</b></summary>

**Timing (~40 min).** 10 min the setup and why it is shocking · 10 min the bulk edges · 10 min the demo · 10 min what it means for PCA.

**Set the trap before running anything.** State the setup precisely: the *true* covariance is the identity, so every true eigenvalue is exactly 1. Now ask what the *sample* covariance's eigenvalues look like with $p = 1000$, $n = 2000$. Most of the room will say "close to 1, with some spread." Then reveal the answer: they fill $[0.09, 2.91]$ — a range of more than 30× — from data whose truth is perfectly isotropic.

**That is the session, and it should feel alarming.** Every one of those eigenvalues is an artifact. A scree plot of this data would show a "dominant component" at 2.9 and a "negligible" one at 0.09, and a researcher would happily interpret both. Neither exists.

**Then the reason, which is a counting argument.** You are estimating $p(p+1)/2 \approx 500{,}000$ covariance parameters from $n \times p = 2$ million numbers. That is only four observations per parameter — nowhere near enough for the classical asymptotics ($n \to \infty$ with $p$ fixed) to apply. Random matrix theory is the correct asymptotic regime when $p$ and $n$ grow *together*, and $\gamma = p/n$ is the parameter that matters. Emphasise that classical statistics is not wrong here, it is simply answering a different limit.

**The edges are the memorable result.** $[(1-\sqrt\gamma)^2, (1+\sqrt\gamma)^2]$. Have the room evaluate it for a few ratios: $\gamma = 0.01$ gives $[0.81, 1.21]$, fairly tight; $\gamma = 0.5$ gives $[0.09, 2.91]$; $\gamma = 1$ gives $[0, 4]$, with eigenvalues reaching zero. That progression makes "high-dimensional" concrete — it is not about $p$ being large, it is about $p/n$ not being small.

**Ask the room.** "So how do I know whether a PCA component is real?" Compare it against the upper bulk edge for your $\gamma$. Anything inside the bulk is structurally indistinguishable from noise, however large it looks relative to its neighbours. That single check is the most immediately usable thing in this workshop, and most practitioners have never heard of it.

**Close by naming where it bites.** Portfolio covariance estimation, gene expression studies, fMRI, and — closest to home — [array processing](../../Intro_DSP/Array_Processing.ipynb), where "how many snapshots do I need?" is exactly the question of making $\gamma$ small enough. Session 3 turns that into a threshold.
</details>

## 3. The Noise Bulk

💡 **Intuition.** Estimate a covariance from $n$ samples of $p$-dimensional *white* noise (true covariance $= I$: all eigenvalues 1). With $p/n = \gamma$ not small, the sample eigenvalues **spread** across $[(1-\sqrt\gamma)^2, (1+\sqrt\gamma)^2]$ — the Marchenko–Pastur bulk. At $p/n = 1/2$, 'eigenvalues' of pure noise range from 0.09 to 2.9! Every PCA scree plot with $p \sim n$ contains this artifact, and everything inside the bulk is *structurally indistinguishable from noise*.

In [3]:
# ORACLE: empirical bulk edges vs the analytic (1 ± √γ)²   [γ = p/n]
p, n_samp = 1000, 2000
gamma = p / n_samp
X = rng.standard_normal((n_samp, p))
S = X.T @ X / n_samp
eigs = np.linalg.eigvalsh(S)

lo, hi = (1 - np.sqrt(gamma))**2, (1 + np.sqrt(gamma))**2
x = np.linspace(lo, hi, 400)
mp = np.sqrt(np.maximum((hi - x) * (x - lo), 0)) / (2*np.pi*gamma*x)
plt.figure(figsize=(7.5, 2.8))
plt.hist(eigs, bins=80, density=True, alpha=0.6, label="sample covariance of PURE NOISE")
plt.plot(x, mp, "k", linewidth=2, label="Marchenko–Pastur density")
for e in (lo, hi): plt.axvline(e, color="r", linestyle=":", linewidth=1)
plt.legend(); plt.title(f"true covariance = I, yet eigenvalues fill [{lo:.2f}, {hi:.2f}]")
plt.tight_layout(); plt.show()
print(f"analytic edges ({lo:.4f}, {hi:.4f})   empirical (min, max) = ({eigs.min():.4f}, {eigs.max():.4f})")
assert abs(eigs.max() - hi) < 0.05 and abs(eigs.min() - lo) < 0.05

analytic edges (0.0858, 2.9142)   empirical (min, max) = (0.0871, 2.9266)


/tmp/ipykernel_2981573/321660279.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** The true covariance is the **identity** — every true eigenvalue is exactly 1. The *sample* covariance's eigenvalues fill $[0.087, 2.927]$, matching the analytic Marchenko–Pastur edges $(0.0858, 2.9142)$ to two decimal places, with the `assert` enforcing agreement to 0.05.

**Read that again, because it is alarming.** From data with perfectly isotropic truth, we obtained a spread of more than **30×** between the smallest and largest sample eigenvalue. A scree plot of this data would show a commanding "first component" at 2.9 and a "negligible" direction at 0.09. **Both are artifacts.** There is no structure whatsoever in the underlying distribution.

**Why classical intuition fails here.** We estimated $p(p+1)/2 \approx 500{,}000$ covariance parameters from $n \times p = 2$ million numbers — about four observations per parameter. Classical asymptotics assume $n \to \infty$ with $p$ *fixed*, which is simply not the regime we are in. Random matrix theory is the correct asymptotic when $p$ and $n$ grow together, and the governing parameter is $\gamma = p/n$. Classical statistics is not wrong; it is answering a different question.

**The edge formula is the thing to memorise:** $[(1-\sqrt\gamma)^2, (1+\sqrt\gamma)^2]$. Evaluate it at a few ratios and "high-dimensional" stops being vague:

| $\gamma = p/n$ | noise bulk |
|---|---|
| 0.01 | [0.81, 1.21] |
| 0.25 | [0.25, 2.25] |
| 0.50 | [0.09, 2.91] |
| 1.00 | [0.00, 4.00] |

The problem is not that $p$ is large. It is that $p/n$ is not small. Ten thousand samples of a thousand-dimensional variable is $\gamma = 0.1$ and still visibly spread.

**Which gives you an immediately usable check.** Before believing any principal component, compute $(1+\sqrt{p/n})^2$ for your data and ask whether the eigenvalue exceeds it. **Anything inside the bulk is structurally indistinguishable from noise**, no matter how dominant it looks relative to its neighbours. Most practitioners have never applied this test, and it invalidates a meaningful fraction of published scree-plot interpretation in genomics, finance, and neuroimaging.

Closest to home, this is the quantitative form of "how many snapshots does [MUSIC](../../Intro_DSP/Array_Processing.ipynb) need?" — enough that $\gamma$ is small enough for real sources to clear the bulk. Session 3 makes that a sharp threshold.

---
### 🕐 Session 3 of 3 — *Spiked Models & the Detection Threshold* (~40 min)
**Goal:** when does a real signal's eigenvalue escape the noise bulk? The BBP phase transition.
**Builds on:** Session 2.

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Spiked Models & the Detection Threshold</b></summary>

**Timing (~40 min).** 8 min the setup · 12 min the BBP threshold · 12 min the demo, especially the right-hand panel · 8 min what it means for array processing.

**Frame it as the question every practitioner actually has.** Session 2 said noise fills a predictable bulk. Now add *one* real signal of strength $\theta$ — a source hitting an [array](../../Intro_DSP/Array_Processing.ipynb), a true factor in returns. Does it show up as a large eigenvalue? The honest answer is: **only above a threshold**, and below it the signal is not weak but *invisible*.

**Insist that this is a phase transition, not a fade.** Above $\theta = \sqrt\gamma$ the top eigenvalue detaches from the bulk and follows $(1+\theta)(1+\gamma/\theta)$ exactly. Below it, the top eigenvalue sits *at the bulk edge* and tells you nothing. There is no intermediate regime where detection is "hard but possible" by spectral means — the transition is sharp. Students trained on "more SNR is gradually better" find this genuinely surprising, and it is the most important idea in the workshop.

**The right-hand panel matters more than the left — do not let it be an afterthought.** Eigenvalue *position* above the threshold is nice; eigenvector *overlap* is the practical quantity, because that is the estimated direction MUSIC or PCA would report. Below threshold the overlap is 0.034, and the number to compare it against is the random-guess baseline: for a unit vector in $p = 400$ dimensions, $E|u^\top v| \approx \sqrt{2/\pi p} = 0.040$. So **0.034 is not "small overlap" — it is exactly what you get from a vector containing no information at all.** Have the room compute that baseline; converting "≈ 0" into "at the random-guess floor" is what makes the claim rigorous rather than impressionistic.

**Then convert the threshold into engineering advice.** $\theta > \sqrt{p/n}$ rearranges to $n > p/\theta^2$. For a 100-element array and a spike of strength 0.5, that is 400 snapshots — a concrete, checkable requirement. This is the quantitative form of the vague "MUSIC needs enough snapshots," and it is arguably the single most useful number in this workshop for anyone doing array work.

**Be honest about the demo's precision.** Max deviation from the BBP formula above threshold is 0.191, which is not tiny. It is finite-$p$ fluctuation: each point is one draw at $p = 400$, and BBP is an asymptotic law. Averaging over draws or raising $p$ would tighten it. Say so — the qualitative shape (flat below, rising above, following the curve) is the verified claim; the pointwise agreement is loose.

**Close on the consequence for practice.** Below threshold, no eigenvalue method sees the signal — not MUSIC, not PCA, not any cleverer spectral algorithm, because the information is not in the spectrum. Recovering it requires *additional structure*: sparsity, known waveform, temporal correlation. That is precisely why [compressed sensing](../../Intro_DSP/Compressed_Sensing.ipynb) and matched filtering exist alongside subspace methods.
</details>

## 4. The Phase Transition

💡 **Intuition.** Add one rank-one signal of strength $\theta$ to the noise (a source hitting an [array](../../Intro_DSP/Array_Processing.ipynb)). Does the top sample eigenvalue reveal it? **Only above a threshold**: for $\theta > \sqrt{\gamma}$ the top eigenvalue pops out of the bulk at $(1+\theta)(1+\gamma/\theta)$; below it, the spike is *swallowed* — no eigenvalue method can see it, however clever (the BBP transition). This is the sharp version of 'how many snapshots do I need': MUSIC's source count, PCA's component count, all gated by $\theta \gtrless \sqrt{p/n}$.

In [4]:
# sweep the spike strength through the threshold — ORACLE: the BBP position formula
p, n_samp = 400, 1600
gamma = p / n_samp                                     # √γ = 0.5
u = rng.standard_normal(p); u /= np.linalg.norm(u)
thetas = np.linspace(0.05, 1.6, 25)
top_eigs, overlaps = [], []
for theta in thetas:
    X = rng.standard_normal((n_samp, p)) @ np.linalg.cholesky(np.eye(p) + theta*np.outer(u, u)).T
    S = X.T @ X / n_samp
    w, V = np.linalg.eigh(S)
    top_eigs.append(w[-1]); overlaps.append(abs(V[:, -1] @ u))

bulk_edge = (1 + np.sqrt(gamma))**2
bbp = [(1+t)*(1+gamma/t) if t > np.sqrt(gamma) else bulk_edge for t in thetas]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3))
axes[0].plot(thetas, top_eigs, "o", markersize=4, label="top eigenvalue (empirical)")
axes[0].plot(thetas, bbp, "k-", linewidth=1.2, label="BBP prediction")
axes[0].axvline(np.sqrt(gamma), color="r", linestyle=":", label="threshold √γ")
axes[0].axhline(bulk_edge, color="gray", linestyle=":", linewidth=0.8)
axes[0].legend(fontsize=7); axes[0].set_xlabel("spike strength θ"); axes[0].set_title("eigenvalue escape")
axes[1].plot(thetas, overlaps, "o-", markersize=4)
axes[1].axvline(np.sqrt(gamma), color="r", linestyle=":")
axes[1].set_xlabel("spike strength θ"); axes[1].set_title("eigenvector overlap with truth:\nzero below threshold — not weak, ZERO")
plt.tight_layout(); plt.show()

pred_err = np.abs(np.array(top_eigs)[thetas > 0.7] - np.array(bbp)[thetas > 0.7]).max()
print(f"max |top eig − BBP formula| above threshold: {pred_err:.3f}")
print(f"below threshold, eigenvector overlap ≈ {np.mean([o for t, o in zip(thetas, overlaps) if t < 0.35]):.3f} — the signal is invisible")

max |top eig − BBP formula| above threshold: 0.191
below threshold, eigenvector overlap ≈ 0.034 — the signal is invisible


/tmp/ipykernel_2981573/3987407102.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Sweeping the spike strength $\theta$ through the threshold $\sqrt\gamma = 0.5$ produces a **phase transition**, not a gradual improvement. Below it the top eigenvalue sits flat at the bulk edge; above it, it detaches and tracks the BBP prediction $(1+\theta)(1+\gamma/\theta)$.

**The right-hand panel is the one that matters, and its number needs a baseline.** Below threshold the eigenvector overlap with the true direction is **0.034**. That sounds small; the question is *how* small. For a random unit vector in $p = 400$ dimensions, the expected overlap with any fixed direction is $\sqrt{2/\pi p} = \mathbf{0.040}$.

So 0.034 is not "weak recovery" — it is **exactly the random-guess floor**. The estimated eigenvector below threshold carries no information about the signal whatsoever. It is not a degraded answer to be improved with better processing; it is a vector pointing nowhere in particular, and reporting it as a direction estimate would be reporting noise.

**Why this is a threshold and not a fade.** Above $\sqrt\gamma$ the signal's eigenvalue escapes the noise bulk and becomes visible; below it, the signal's eigenvalue is *inside* the bulk, where Session 2 established that everything is structurally indistinguishable from noise. There is no intermediate regime where detection is difficult but possible by spectral means. The transition is sharp — which is a real surprise for anyone trained to expect that more SNR is gradually better.

**Turn the threshold into a snapshot budget, because that is what it is for.** $\theta > \sqrt{p/n}$ rearranges to
$$n > \frac{p}{\theta^2}.$$
A 100-element array with a spike of strength 0.5 needs more than 400 snapshots. That is the quantitative version of the usual hand-wave that "[MUSIC](../../Intro_DSP/Array_Processing.ipynb) needs enough snapshots," and it is checkable before you build anything. The same inequality governs how many observations a factor model needs before its factors are real.

**Be honest about the fit.** Maximum deviation from BBP above threshold is **0.191** — not tight. Each point is a single draw at $p = 400$, and BBP is an asymptotic law, so finite-size fluctuation dominates; averaging over draws or increasing $p$ would shrink it considerably. The verified claim here is the *shape* — flat below threshold, detaching above, following the predicted curve — rather than pointwise agreement.

**And the consequence worth carrying away.** Below threshold, **no eigenvalue method can see the signal.** Not MUSIC, not PCA, not any more sophisticated subspace algorithm — because the information is genuinely absent from the spectrum rather than merely hard to extract. Recovering it requires bringing in structure the spectrum does not use: sparsity ([compressed sensing](../../Intro_DSP/Compressed_Sensing.ipynb)), a known waveform (matched filtering), or temporal correlation. Knowing where the wall is tells you when to stop tuning your eigen-solver and start changing your assumptions.

## 5. Conclusion

Random eigenvalues obey deterministic laws; pure noise fills a predictable bulk (edges verified to 2 decimals); and signals are detectable by spectra *only* above $\sqrt{p/n}$ — a phase transition, not a gradual fade. Check every scree plot against Marchenko–Pastur before believing a single 'component'.

---
## Where next

- [Array Processing](../../Intro_DSP/Array_Processing.ipynb) — MUSIC's snapshot budget, now quantitative.
- [Numerical Linear Algebra](../Numerical_Linear_Algebra/Numerical_Linear_Algebra.ipynb) — why random sketches capture ranges.
- [Concentration](../Concentration/Concentration_Inequalities.ipynb) — the self-averaging machinery.